# Checkpoint 1200 Dual-Branch Pair-Priority Decoder Eval

Evaluate `checkpoint-1200` with the dual-branch pair-priority conditional decoder.


In [ ]:
# 1) Install dependencies, then restart runtime once.
# After restart, run this cell again and continue.
import os
import subprocess
import sys
from pathlib import Path

MARKER = Path("/content/.snu_pilot_c_deps_installed")

if Path("/content").exists() and not MARKER.exists():
    packages = [
        "transformers>=4.49.0,<4.54.0",
        "accelerate>=0.34.0",
        "bitsandbytes>=0.46.1",
        "peft",
        "qwen-vl-utils",
        "huggingface_hub",
        "hf_xet",
        "modelscope",
        "jedi",
        "pandas==2.2.2",
        "safetensors>=0.4.5",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])
    MARKER.write_text("ok")
    print("Dependencies installed. Restarting runtime. Run this cell again after restart.")
    os.kill(os.getpid(), 9)
elif Path("/content").exists():
    print("Dependencies already installed. Continue.")
else:
    print("Local environment detected. Skipping Colab dependency install.")


In [ ]:
# 2) Setup: Drive, data, model cache, run paths
from google.colab import drive
from pathlib import Path
import os
import shutil

drive_root = Path("/content/drive")
if drive_root.exists() and not os.path.ismount(str(drive_root)) and any(drive_root.iterdir()):
    print("Removing local pre-mount /content/drive contents:", sorted(str(p) for p in drive_root.iterdir())[:20])
    shutil.rmtree(drive_root)
drive_root.mkdir(parents=True, exist_ok=True)
drive.mount("/content/drive")

import ast
import copy
import gc
import glob
import itertools
import json
import math
import random
import re
import subprocess
import zipfile
from datetime import datetime

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig, Trainer, TrainingArguments, TrainerCallback, set_seed
try:
    from transformers import Qwen2VLForConditionalGeneration
except ImportError:
    Qwen2VLForConditionalGeneration = AutoModelForVision2Seq
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers.utils import logging as transformers_logging

transformers_logging.set_verbosity_error()

SNU_ROOT = Path("/content/drive/MyDrive/SNU_AI_Challenge")
assert SNU_ROOT.exists(), SNU_ROOT

ZIP_PATH = SNU_ROOT / "snuaichallenge.zip"
DATA_DIR = Path("/content/snuaichallenge_data")
TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV = DATA_DIR / "test.csv"
TRAIN_IMAGE_DIR = DATA_DIR / "train"
TEST_IMAGE_DIR = DATA_DIR / "test"

if not TRAIN_CSV.exists() or not TRAIN_IMAGE_DIR.is_dir():
    print("Extracting:", ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as zip_file:
        zip_file.extractall("/content/")

assert TRAIN_CSV.exists(), TRAIN_CSV
assert TEST_CSV.exists(), TEST_CSV
assert TRAIN_IMAGE_DIR.is_dir(), TRAIN_IMAGE_DIR
assert TEST_IMAGE_DIR.is_dir(), TEST_IMAGE_DIR

MODEL_REPO_ID = "Qwen/Qwen2-VL-7B-Instruct"
USE_MODELSCOPE_BASE_MODEL = False
DRIVE_MODEL_DIR = SNU_ROOT / "model_cache/Qwen2-VL-7B-Instruct"
LOCAL_MODEL_DIR = Path("/content/Qwen2-VL-7B-Instruct")


def print_runtime_storage():
    print("Storage check for /content:")
    try:
        subprocess.run(["df", "-h", "/content"], check=False)
    except Exception as exc:
        total, used, free = shutil.disk_usage("/content")
        print(f"/content free: {free / (1024 ** 3):.1f} GB / total: {total / (1024 ** 3):.1f} GB ({exc})")

    try:
        meminfo = {}
        with open("/proc/meminfo", "r", encoding="utf-8") as handle:
            for line in handle:
                key, value = line.split(":", 1)
                meminfo[key] = int(value.strip().split()[0]) / (1024 ** 2)
        print(f"RAM available: {meminfo.get('MemAvailable', 0):.1f} GB / total: {meminfo.get('MemTotal', 0):.1f} GB")
    except Exception as exc:
        print("RAM check skipped:", exc)

    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        print(f"GPU memory free: {free / (1024 ** 3):.1f} GB / total: {total / (1024 ** 3):.1f} GB")


def model_cache_is_complete(model_dir):
    model_dir = Path(model_dir)
    if not (model_dir / "config.json").exists():
        return False
    has_weight = (model_dir / "model.safetensors.index.json").exists() or bool(list(model_dir.glob("*.safetensors")))
    has_processor = any((model_dir / name).exists() for name in ["preprocessor_config.json", "processor_config.json", "tokenizer.json", "tokenizer_config.json"])
    return bool(has_weight and has_processor)


def copy_drive_cache_to_local():
    if not model_cache_is_complete(DRIVE_MODEL_DIR):
        raise FileNotFoundError(f"Drive model cache is incomplete or missing: {DRIVE_MODEL_DIR}")

    if model_cache_is_complete(LOCAL_MODEL_DIR):
        print("Using existing local model:", LOCAL_MODEL_DIR)
        return

    print("Copying base model from Drive to Colab local disk...")
    print("  from:", DRIVE_MODEL_DIR)
    print("  to  :", LOCAL_MODEL_DIR)
    if LOCAL_MODEL_DIR.exists():
        shutil.rmtree(LOCAL_MODEL_DIR)
    shutil.copytree(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR)
    assert model_cache_is_complete(LOCAL_MODEL_DIR), LOCAL_MODEL_DIR


def download_base_model_to_local_and_cache():
    print("Base model cache not found. Downloading with Hugging Face snapshot_download to local disk first:")
    print("repo:", MODEL_REPO_ID)
    if LOCAL_MODEL_DIR.exists() and not model_cache_is_complete(LOCAL_MODEL_DIR):
        shutil.rmtree(LOCAL_MODEL_DIR)
    from huggingface_hub import snapshot_download
    hf_token = os.environ.get("HF_TOKEN")
    model_dir = snapshot_download(
        repo_id=MODEL_REPO_ID,
        local_dir=str(LOCAL_MODEL_DIR),
        token=hf_token,
        max_workers=8,
    )
    print("Downloaded:", model_dir)
    assert model_cache_is_complete(LOCAL_MODEL_DIR), LOCAL_MODEL_DIR

    DRIVE_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(DRIVE_MODEL_DIR) + ".tmp")
    if tmp.exists():
        shutil.rmtree(tmp)
    shutil.copytree(LOCAL_MODEL_DIR, tmp)
    if DRIVE_MODEL_DIR.exists():
        shutil.rmtree(DRIVE_MODEL_DIR)
    os.replace(tmp, DRIVE_MODEL_DIR)
    print("Saved to Drive:", DRIVE_MODEL_DIR)


def ensure_base_model_path():
    print_runtime_storage()

    if model_cache_is_complete(LOCAL_MODEL_DIR):
        print("Using local model:", LOCAL_MODEL_DIR)
    elif model_cache_is_complete(DRIVE_MODEL_DIR):
        copy_drive_cache_to_local()
    elif not USE_MODELSCOPE_BASE_MODEL:
        download_base_model_to_local_and_cache()
    else:
        print("Base model cache not found. Downloading via ModelScope:", MODEL_REPO_ID)
        from modelscope import snapshot_download as modelscope_snapshot_download
        model_dir = modelscope_snapshot_download(MODEL_REPO_ID, cache_dir="/content/modelscope_cache")
        if LOCAL_MODEL_DIR.exists():
            shutil.rmtree(LOCAL_MODEL_DIR)
        shutil.copytree(model_dir, LOCAL_MODEL_DIR)
        assert model_cache_is_complete(LOCAL_MODEL_DIR), LOCAL_MODEL_DIR
        DRIVE_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
        tmp = Path(str(DRIVE_MODEL_DIR) + ".tmp")
        if tmp.exists():
            shutil.rmtree(tmp)
        shutil.copytree(LOCAL_MODEL_DIR, tmp)
        if DRIVE_MODEL_DIR.exists():
            shutil.rmtree(DRIVE_MODEL_DIR)
        os.replace(tmp, DRIVE_MODEL_DIR)
        print("Saved to Drive:", DRIVE_MODEL_DIR)

    print_runtime_storage()
    print("Using local model:", LOCAL_MODEL_DIR)
    return str(LOCAL_MODEL_DIR)


MODEL_ID = ensure_base_model_path()
MODEL_LOCAL_FILES_ONLY = True

OUTPUT_ROOT = SNU_ROOT / "qwen2vl_7b_multitask_bipair_conditional_v1"
# Set this to a specific run id when needed. If None, the latest run is used.
PILOT_RUN_ID = "20260721_011312"

def resolve_run_root(output_root, run_id=None):
    runs_root = output_root / "runs"
    if run_id is not None:
        run_root = runs_root / run_id
        assert run_root.is_dir(), run_root
        return run_root
    candidates = sorted([p for p in runs_root.iterdir() if p.is_dir()])
    if not candidates:
        raise RuntimeError(f"No runs found under {runs_root}")
    return candidates[-1]

RUN_ROOT = resolve_run_root(OUTPUT_ROOT, PILOT_RUN_ID)
RUN_ID = RUN_ROOT.name
OUTPUT_DIR = RUN_ROOT / "multitask_bipair_conditional"
EVAL_DIR = OUTPUT_DIR / "eval"
BEST_ADAPTER_DIR = OUTPUT_DIR / "best_adapter"
SUBMIT_PATH = OUTPUT_DIR / "submission_pilot_c.csv"
for path in [EVAL_DIR, BEST_ADAPTER_DIR]:
    path.mkdir(parents=True, exist_ok=True)

SEED = 42
VALID_RATIO = 0.10
TRAIN_ROWS = None
VALID_ROWS = None
FULL_EVAL_ROWS = 100
SELECTED_CHECKPOINTS = ["checkpoint-1200"]
SCORE_BATCH_SIZE = 8
NEXT_TOKEN_BATCH_SIZE = 16
RUN_DIRECT_GENERATION = False
USE_KV_CACHE_SEQUENCE_SCORING = True

TASK_RATIOS = {
    "order": 0.30,
    "pairwise": 0.25,
    "first": 0.10,
    "last": 0.10,
    "fixed_first": 0.10,
    "fixed_last": 0.10,
    "fixed_endpoints": 0.05,
}
TASK_LOSS_WEIGHTS = {task: 1.0 for task in TASK_RATIOS}

LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

LEARNING_RATE = 1e-5
NUM_TRAIN_EPOCHS = 1
MAX_TRAIN_STEPS = -1
SAVE_STEPS = 100
LOGGING_STEPS = 20

MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 256 * 28 * 28
PAIR_INDICES = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
PERMUTATIONS = list(itertools.permutations([1, 2, 3, 4]))

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("run root:", RUN_ROOT)
print("output:", OUTPUT_DIR)
print("model:", MODEL_ID)


In [ ]:
# 3) Data split, prompts, metrics, and model helpers
def parse_answer(answer):
    result = answer if isinstance(answer, list) else ast.literal_eval(str(answer))
    result = [int(value) for value in result]
    if len(result) != 4 or sorted(result) != [1, 2, 3, 4]:
        raise ValueError(f"Invalid Answer: {answer}")
    return result


def order_to_sequence(answer):
    return [input_index + 1 for input_index, _ in sorted(enumerate(answer), key=lambda item: item[1])]


def compact_order(order):
    return " ".join(str(int(value)) for value in order)


def parse_compact_order(text, expected_len=4):
    values = [int(x) for x in re.findall(r"[1-4]", str(text))]
    if len(values) != expected_len or len(set(values)) != expected_len:
        return None
    return values


def row_image_paths(row, image_root=TRAIN_IMAGE_DIR):
    sample_id = str(row["Id"])
    return [str(Path(image_root) / sample_id / str(row[f"Input_{i}"])) for i in range(1, 5)]


from functools import lru_cache


@lru_cache(maxsize=256)
def _load_rgb_cached(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()


def load_rgb(path):
    return _load_rgb_cached(str(path)).copy()


def base_record(row_index, row):
    answer = [int(value) for value in row["Answer_list"]]
    order = order_to_sequence(answer)
    return {
        "row_index": int(row_index),
        "sample_id": str(row["Id"]),
        "sentence": "" if pd.isna(row["Sentence"]) else str(row["Sentence"]),
        "answer": answer,
        "order": order,
        "image_paths": row_image_paths(row, TRAIN_IMAGE_DIR),
    }


def pair_target_for_order(order, a, b):
    ranks = {frame: idx for idx, frame in enumerate(order)}
    return "A" if ranks[int(a)] < ranks[int(b)] else "B"


def build_record_pools(dataframe):
    pools = {task: [] for task in TASK_RATIOS}
    for row_index, row in dataframe.iterrows():
        base = base_record(row_index, row)
        order = base["order"]

        item = copy.deepcopy(base)
        item.update({"task_type": "order", "target": compact_order(order)})
        pools["order"].append(item)

        item = copy.deepcopy(base)
        item.update({"task_type": "first", "target": str(order[0])})
        pools["first"].append(item)

        item = copy.deepcopy(base)
        item.update({"task_type": "last", "target": str(order[-1])})
        pools["last"].append(item)

        for i, j in PAIR_INDICES:
            a, b = i + 1, j + 1
            for left, right in [(a, b), (b, a)]:
                item = copy.deepcopy(base)
                item.update({
                    "task_type": "pairwise",
                    "pair": [left, right],
                    "image_paths": [base["image_paths"][left - 1], base["image_paths"][right - 1]],
                    "target": pair_target_for_order(order, left, right),
                })
                pools["pairwise"].append(item)

        first = order[0]
        remaining = [x for x in order if x != first]
        item = copy.deepcopy(base)
        item.update({"task_type": "fixed_first", "fixed_first": first, "target": compact_order(remaining)})
        pools["fixed_first"].append(item)

        last = order[-1]
        remaining = [x for x in order if x != last]
        item = copy.deepcopy(base)
        item.update({"task_type": "fixed_last", "fixed_last": last, "target": compact_order(remaining)})
        pools["fixed_last"].append(item)

        middle = [x for x in order if x not in {order[0], order[-1]}]
        item = copy.deepcopy(base)
        item.update({"task_type": "fixed_endpoints", "fixed_first": order[0], "fixed_last": order[-1], "target": compact_order(middle)})
        pools["fixed_endpoints"].append(item)
    return pools


def sample_records(records, count, rng):
    indices = rng.integers(0, len(records), size=count)
    return [records[int(index)] for index in indices]


def build_balanced_records(dataframe):
    pools = build_record_pools(dataframe)
    base_total = int(math.ceil(len(pools["order"]) / TASK_RATIOS["order"]))
    rng = np.random.default_rng(SEED)
    merged = []
    distribution = {}
    for task, ratio in TASK_RATIOS.items():
        count = max(1, int(round(base_total * ratio)))
        records = sample_records(pools[task], count, rng)
        merged.extend(records)
        distribution[task] = {"pool": len(pools[task]), "sampled": len(records)}
        print(task, distribution[task])
    rng.shuffle(merged)
    return merged, pools, distribution


train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
train_df["Id"] = train_df["Id"].astype(str)
test_df["Id"] = test_df["Id"].astype(str)
train_df["Answer_list"] = train_df["Answer"].apply(parse_answer)

split_root = SNU_ROOT / "id_splits" / "qwen2vl_lgt_order_refine_20260714_003635"
if (split_root / "train_ids.json").exists() and (split_root / "validation_ids.json").exists():
    train_ids = set(str(x) for x in json.load(open(split_root / "train_ids.json", "r", encoding="utf-8")))
    valid_ids = set(str(x) for x in json.load(open(split_root / "validation_ids.json", "r", encoding="utf-8")))
else:
    unique_ids = train_df["Id"].unique().copy()
    rng = np.random.default_rng(SEED)
    rng.shuffle(unique_ids)
    valid_size = max(1, int(len(unique_ids) * VALID_RATIO))
    valid_ids = set(unique_ids[:valid_size])
    train_ids = set(unique_ids[valid_size:])

training_df = train_df[train_df["Id"].isin(train_ids)].reset_index(drop=True)
validation_df = train_df[train_df["Id"].isin(valid_ids)].reset_index(drop=True)
if TRAIN_ROWS is not None:
    training_df = training_df.sample(n=min(TRAIN_ROWS, len(training_df)), random_state=SEED).reset_index(drop=True)
if VALID_ROWS is not None:
    validation_df = validation_df.sample(n=min(VALID_ROWS, len(validation_df)), random_state=SEED).reset_index(drop=True)

print("train/valid/test:", len(training_df), len(validation_df), len(test_df))
print("eval rows are sampled directly from validation_df; training record pools are not rebuilt in this eval notebook.")


# Free stale objects before loading the 7B base model. This matters when a previous
# cell was interrupted during shard loading in the same Colab runtime.
for _name in ["trainer", "model", "base_model", "processor"]:
    if _name in globals():
        del globals()[_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"GPU memory before model load: {free / (1024 ** 3):.1f} GB free / {total / (1024 ** 3):.1f} GB total")


processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    local_files_only=MODEL_LOCAL_FILES_ONLY,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
processor.tokenizer.padding_side = "right"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)


def task_instruction(example):
    return globals()["task_instruction_train"](example) if "task_instruction_train" in globals() else task_instruction_impl(example)


def task_instruction_impl(example):
    sentence = example["sentence"]
    task_type = example["task_type"]
    if task_type == "pairwise":
        return f"Caption:\n{sentence}\n\nThe two candidate images are labeled A and B in the presented order.\nWhich image occurs earlier in the story timeline?\nAnswer only A or B."
    if task_type == "first":
        return f"Caption:\n{sentence}\n\nWhich image is the first scene in the story? Answer only one frame number from 1 to 4."
    if task_type == "last":
        return f"Caption:\n{sentence}\n\nWhich image is the last scene in the story? Answer only one frame number from 1 to 4."
    if task_type == "order":
        return f"Caption:\n{sentence}\n\nOrder all four images from earliest to latest in the story.\nAnswer only four frame numbers separated by spaces, for example: 1 2 3 4."
    if task_type == "fixed_first":
        return f"Caption:\n{sentence}\n\nFrame {example['fixed_first']} is fixed as the first scene.\nOrder the remaining frames from earliest to latest.\nAnswer only the remaining frame numbers separated by spaces."
    if task_type == "fixed_last":
        return f"Caption:\n{sentence}\n\nFrame {example['fixed_last']} is fixed as the last scene.\nOrder the remaining frames from earliest to latest.\nAnswer only the remaining frame numbers separated by spaces."
    if task_type == "fixed_endpoints":
        return f"Caption:\n{sentence}\n\nFrame {example['fixed_first']} is fixed as the first scene.\nFrame {example['fixed_last']} is fixed as the last scene.\nOrder the remaining middle frames from earliest to latest.\nAnswer only the remaining frame numbers separated by spaces."
    raise ValueError(task_type)


def make_messages(example, include_answer=False):
    content = []
    for idx, _ in enumerate(example["image_paths"], start=1):
        label = "A" if example["task_type"] == "pairwise" and idx == 1 else "B" if example["task_type"] == "pairwise" and idx == 2 else str(idx)
        content.append({"type": "text", "text": f"\nImage {label}:"})
        content.append({"type": "image"})
    content.append({"type": "text", "text": "\n\n" + task_instruction_impl(example)})
    messages = [{"role": "user", "content": content}]
    if include_answer:
        messages.append({"role": "assistant", "content": str(example["target"])})
    return messages


def checkpoint_name(path):
    return Path(path).name


def find_checkpoint_dirs():
    dirs = []
    for path in OUTPUT_DIR.iterdir():
        if path.name.startswith("checkpoint-") and (path / "adapter_config.json").exists():
            dirs.append(path)
    final_dir = OUTPUT_DIR / "final_adapter"
    if (final_dir / "adapter_config.json").exists():
        dirs.append(final_dir)
    def sort_key(path):
        match = re.findall(r"checkpoint-(\d+)", str(path))
        return int(match[-1]) if match else 10**9
    return sorted(dict.fromkeys(dirs), key=sort_key)


EVAL_MODEL = None
CURRENT_ADAPTER_NAME = None


def adapter_name_for(adapter_dir):
    return re.sub(r"[^0-9a-zA-Z_]+", "_", checkpoint_name(adapter_dir))


def load_eval_model(adapter_dir):
    global EVAL_MODEL, CURRENT_ADAPTER_NAME
    adapter_dir = str(adapter_dir)
    adapter_name = adapter_name_for(adapter_dir)

    def load_fresh():
        base = Qwen2VLForConditionalGeneration.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            torch_dtype=torch.float16,
            device_map="auto",
            local_files_only=MODEL_LOCAL_FILES_ONLY,
            trust_remote_code=True,
        low_cpu_mem_usage=True,
        )
        from peft import PeftModel
        return PeftModel.from_pretrained(base, adapter_dir, adapter_name=adapter_name, is_trainable=False)

    if EVAL_MODEL is None:
        EVAL_MODEL = load_fresh()
    else:
        try:
            if CURRENT_ADAPTER_NAME is not None and hasattr(EVAL_MODEL, "delete_adapter"):
                EVAL_MODEL.delete_adapter(CURRENT_ADAPTER_NAME)
            EVAL_MODEL.load_adapter(adapter_dir, adapter_name=adapter_name, is_trainable=False)
            EVAL_MODEL.set_adapter(adapter_name)
        except Exception as exc:
            print("Adapter switch failed; reloading base model:", repr(exc))
            del EVAL_MODEL
            gc.collect()
            torch.cuda.empty_cache()
            EVAL_MODEL = load_fresh()

    CURRENT_ADAPTER_NAME = adapter_name
    EVAL_MODEL.eval()
    if hasattr(EVAL_MODEL, "generation_config"):
        EVAL_MODEL.generation_config.do_sample = False
        EVAL_MODEL.generation_config.temperature = None
        EVAL_MODEL.generation_config.top_p = None
        EVAL_MODEL.generation_config.top_k = None
        EVAL_MODEL.generation_config.num_beams = 1
    return EVAL_MODEL

def model_device(active_model):
    return next(active_model.parameters()).device


def single_token_id(value):
    ids = processor.tokenizer.encode(str(value), add_special_tokens=False)
    if len(ids) != 1:
        raise ValueError(f"{value!r} tokenized to {ids}")
    return ids[0]


DIGIT_TOKEN_IDS = {digit: single_token_id(str(digit)) for digit in [1, 2, 3, 4]}
AB_TOKEN_IDS = {"A": single_token_id("A"), "B": single_token_id("B")}


def make_eval_example(row, task_type, pair=None, fixed_first=None, fixed_last=None, image_root=TRAIN_IMAGE_DIR):
    answer = [int(value) for value in row.get("Answer_list", [1, 2, 3, 4])]
    image_paths = row_image_paths(row, image_root)
    example = {
        "sample_id": str(row["Id"]),
        "sentence": "" if pd.isna(row["Sentence"]) else str(row["Sentence"]),
        "answer": answer,
        "order": order_to_sequence(answer),
        "image_paths": image_paths,
        "task_type": task_type,
        "target": "1",
    }
    if task_type == "pairwise":
        a, b = pair
        example["pair"] = [a, b]
        example["image_paths"] = [image_paths[a - 1], image_paths[b - 1]]
    if fixed_first is not None:
        example["fixed_first"] = int(fixed_first)
    if fixed_last is not None:
        example["fixed_last"] = int(fixed_last)
    return example


In [ ]:
# 4) Scoring and conditional sequential decoding
@torch.no_grad()
def _batch_to_device(inputs, active_model):
    return {key: value.to(model_device(active_model)) if torch.is_tensor(value) else value for key, value in inputs.items()}


@torch.no_grad()
def score_next_token_examples(active_model, examples, token_ids, batch_size=None):
    batch_size = int(batch_size or NEXT_TOKEN_BATCH_SIZE)
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "right"
    ids = list(token_ids.values())
    results = []
    try:
        for start_index in range(0, len(examples), batch_size):
            batch_examples = examples[start_index:start_index + batch_size]
            texts = [
                processor.apply_chat_template(
                    make_messages(example, include_answer=False),
                    tokenize=False,
                    add_generation_prompt=True,
                )
                for example in batch_examples
            ]
            batch_images = [[load_rgb(path) for path in example["image_paths"]] for example in batch_examples]
            inputs = processor(text=texts, images=batch_images, padding=True, return_tensors="pt")
            inputs = _batch_to_device(inputs, active_model)
            outputs = active_model(**inputs)
            for row_index in range(len(batch_examples)):
                last_pos = int(inputs["attention_mask"][row_index].sum().item()) - 1
                logits = outputs.logits[row_index, last_pos]
                probs = torch.softmax(logits[ids].float(), dim=-1).detach().cpu().numpy()
                results.append({key: float(prob) for key, prob in zip(token_ids.keys(), probs)})
    finally:
        processor.tokenizer.padding_side = old_padding_side
    return results


@torch.no_grad()
def score_next_token_candidates(active_model, example, token_ids):
    return score_next_token_examples(active_model, [example], token_ids, batch_size=1)[0]


@torch.no_grad()
def generate_text(active_model, example, max_new_tokens=16):
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "left"
    try:
        text = processor.apply_chat_template(make_messages(example, include_answer=False), tokenize=False, add_generation_prompt=True)
        images = [load_rgb(path) for path in example["image_paths"]]
        inputs = processor(text=[text], images=[images], return_tensors="pt")
        inputs = _batch_to_device(inputs, active_model)
        generated = active_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None, top_k=None)
        output = processor.tokenizer.batch_decode(generated[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
        return output.strip()
    finally:
        processor.tokenizer.padding_side = old_padding_side


@torch.no_grad()
def _score_sequence_candidates_recompute(active_model, example, candidate_orders, batch_size=None):
    batch_size = int(batch_size or SCORE_BATCH_SIZE)
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "right"
    try:
        prompt = processor.apply_chat_template(make_messages(example, include_answer=False), tokenize=False, add_generation_prompt=True)
        images = [load_rgb(path) for path in example["image_paths"]]
        prompt_inputs = processor(text=[prompt], images=[images], return_tensors="pt")
        prompt_len = int(prompt_inputs["attention_mask"][0].sum().item())
        scores = {}
        orders = list(candidate_orders)
        for start_index in range(0, len(orders), batch_size):
            batch_orders = orders[start_index:start_index + batch_size]
            texts = [prompt + compact_order(order) for order in batch_orders]
            batch_images = [images for _ in batch_orders]
            inputs = processor(text=texts, images=batch_images, padding=True, return_tensors="pt")
            inputs = _batch_to_device(inputs, active_model)
            outputs = active_model(**inputs)
            for row_index, order in enumerate(batch_orders):
                input_ids = inputs["input_ids"][row_index]
                attention_len = int(inputs["attention_mask"][row_index].sum().item())
                target_len = attention_len - prompt_len
                target_ids = input_ids[prompt_len:prompt_len + target_len]
                logits = outputs.logits[row_index, prompt_len - 1:prompt_len - 1 + target_len]
                log_probs = torch.log_softmax(logits.float(), dim=-1)
                scores[" ".join(map(str, order))] = float(log_probs.gather(1, target_ids[:, None]).mean().item())
        return scores
    finally:
        processor.tokenizer.padding_side = old_padding_side


@torch.no_grad()
def _target_ids_after_prompt(prompt, answer_text, device):
    tokenizer = processor.tokenizer
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(prompt + answer_text, add_special_tokens=False)["input_ids"]

    if full_ids[:len(prompt_ids)] != prompt_ids:
        raise RuntimeError(
            "Prompt/answer boundary tokenization changed. "
            f"prompt_tail={prompt_ids[-10:]}, "
            f"full_prefix_tail={full_ids[max(0, len(prompt_ids) - 10):len(prompt_ids)]}"
        )

    target_ids = full_ids[len(prompt_ids):]
    if not target_ids:
        raise RuntimeError(f"No target tokens for answer: {answer_text!r}")

    return torch.tensor(target_ids, dtype=torch.long, device=device)


@torch.no_grad()
def build_qwen2vl_continuation_position_ids(prompt_attention_mask, rope_deltas, continuation_length, device):
    if rope_deltas is None:
        raise RuntimeError("Prompt output has no rope_deltas.")

    batch_size = prompt_attention_mask.shape[0]
    prompt_text_lengths = prompt_attention_mask.long().sum(dim=1).to(device)
    offsets = torch.arange(continuation_length, dtype=torch.long, device=device)
    text_positions = prompt_text_lengths[:, None] + offsets[None, :]
    rope_deltas = rope_deltas.to(device).reshape(batch_size, 1)
    mrope_positions = text_positions + rope_deltas
    return mrope_positions.unsqueeze(0).expand(3, -1, -1).contiguous()


@torch.no_grad()
def _score_sequence_candidates_with_cache(active_model, example, candidate_orders):
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "right"
    prompt_cache = None
    base_cache_length = None

    try:
        device = model_device(active_model)
        prompt = processor.apply_chat_template(
            make_messages(example, include_answer=False),
            tokenize=False,
            add_generation_prompt=True,
        )
        images = [load_rgb(path) for path in example["image_paths"]]
        prompt_inputs = processor(text=[prompt], images=[images], return_tensors="pt")
        prompt_inputs = _batch_to_device(prompt_inputs, active_model)

        prompt_outputs = active_model(**prompt_inputs, use_cache=True)
        prompt_cache = prompt_outputs.past_key_values
        if prompt_cache is None:
            raise RuntimeError("Model did not return past_key_values.")
        if not hasattr(prompt_cache, "get_seq_length"):
            raise RuntimeError(f"Unsupported cache type: {type(prompt_cache)}")
        if not hasattr(prompt_cache, "crop"):
            raise RuntimeError(f"Cache does not support crop(): {type(prompt_cache)}")

        base_cache_length = int(prompt_cache.get_seq_length())
        prompt_attention_mask = prompt_inputs["attention_mask"]
        prompt_len = int(prompt_attention_mask[0].sum().item())
        first_logits = prompt_outputs.logits[0, prompt_len - 1]
        prompt_rope_deltas = getattr(prompt_outputs, "rope_deltas", None)
        scores = {}

        for order in candidate_orders:
            prompt_cache.crop(base_cache_length)
            answer_text = compact_order(order)
            target_ids = _target_ids_after_prompt(prompt, answer_text, device)
            candidate_logits = [first_logits]

            if target_ids.numel() > 1:
                previous_ids = target_ids[:-1].unsqueeze(0)
                continuation_mask = torch.ones(
                    (1, previous_ids.shape[1]),
                    dtype=prompt_attention_mask.dtype,
                    device=device,
                )
                attention_mask = torch.cat([prompt_attention_mask, continuation_mask], dim=1)
                cache_position = torch.arange(
                    base_cache_length,
                    base_cache_length + previous_ids.shape[1],
                    dtype=torch.long,
                    device=device,
                )
                position_ids = build_qwen2vl_continuation_position_ids(
                    prompt_attention_mask=prompt_attention_mask,
                    rope_deltas=prompt_rope_deltas,
                    continuation_length=previous_ids.shape[1],
                    device=device,
                )
                continuation_kwargs = {
                    "input_ids": previous_ids,
                    "attention_mask": attention_mask,
                    "position_ids": position_ids,
                    "past_key_values": prompt_cache,
                    "cache_position": cache_position,
                    "use_cache": True,
                    "return_dict": True,
                    "pixel_values": None,
                    "pixel_values_videos": None,
                    "image_grid_thw": None,
                    "video_grid_thw": None,
                }

                continuation_outputs = active_model(**continuation_kwargs)
                candidate_logits.extend(
                    continuation_outputs.logits[0, idx]
                    for idx in range(continuation_outputs.logits.shape[1])
                )

            logits = torch.stack(candidate_logits[:target_ids.numel()], dim=0)
            log_probs = torch.log_softmax(logits.float(), dim=-1)
            token_log_probs = log_probs.gather(1, target_ids[:, None]).squeeze(1)
            scores[" ".join(map(str, order))] = float(token_log_probs.mean().item())

        prompt_cache.crop(base_cache_length)
        return scores
    finally:
        if prompt_cache is not None and base_cache_length is not None and hasattr(prompt_cache, "crop"):
            prompt_cache.crop(base_cache_length)
        processor.tokenizer.padding_side = old_padding_side


@torch.no_grad()
def score_sequence_candidates(active_model, example, candidate_orders, batch_size=None):
    candidate_orders = list(candidate_orders)
    if USE_KV_CACHE_SEQUENCE_SCORING:
        return _score_sequence_candidates_with_cache(active_model, example, candidate_orders)
    return _score_sequence_candidates_recompute(active_model, example, candidate_orders, batch_size=batch_size)


def softmax_scores(scores, temperature=1.0):
    keys = list(scores.keys())
    values = np.array([scores[k] for k in keys], dtype=np.float64) / temperature
    values = values - values.max()
    probs = np.exp(values)
    probs = probs / probs.sum()
    return {k: float(v) for k, v in zip(keys, probs)}


def parse_order_key(key):
    return tuple(int(x) for x in str(key).split())


def pair_logit(p, eps=1e-6):
    p = min(max(float(p), eps), 1.0 - eps)
    return math.log(p / (1.0 - p))


def combine_bidirectional_pair(p_forward, p_reverse, eps=1e-6):
    combined_logit = 0.5 * (pair_logit(p_forward, eps) - pair_logit(p_reverse, eps))
    return 1.0 / (1.0 + math.exp(-combined_logit))


def normalized(scores):
    total = sum(max(float(v), 0.0) for v in scores.values())
    if total <= 0:
        return {int(k): 1.0 / len(scores) for k in scores}
    return {int(k): max(float(v), 0.0) / total for k, v in scores.items()}


def position_marginal(order_probs, position, candidates):
    out = {int(c): 0.0 for c in candidates}
    for key, prob in order_probs.items():
        order = parse_order_key(key)
        if order[position] in out:
            out[order[position]] += float(prob)
    return normalized(out)


def endpoint_probs_from_pair(pair_probs, candidates, mode):
    candidates = [int(x) for x in candidates]
    if mode == "first":
        return normalized({i: sum(pair_probs[f"{i}>{j}"]["combined_prob"] for j in candidates if j != i) for i in candidates})
    return normalized({i: sum(pair_probs[f"{j}>{i}"]["combined_prob"] for j in candidates if j != i) for i in candidates})


def fuse_three_raw(order_signal, endpoint_signal, pair_signal, w_order, w_endpoint, w_pair):
    candidates = sorted(set(order_signal) | set(endpoint_signal) | set(pair_signal))
    return {
        int(i): (
            w_order * order_signal.get(i, 0.0)
            + w_endpoint * endpoint_signal.get(i, 0.0)
            + w_pair * pair_signal.get(i, 0.0)
        )
        for i in candidates
    }


def fuse_three(order_signal, endpoint_signal, pair_signal, w_order, w_endpoint, w_pair):
    return normalized(fuse_three_raw(order_signal, endpoint_signal, pair_signal, w_order, w_endpoint, w_pair))


def normalize_endpoint_slots(first_scores, last_scores):
    raw = {f"first:{int(k)}": max(float(v), 0.0) for k, v in first_scores.items()}
    raw.update({f"last:{int(k)}": max(float(v), 0.0) for k, v in last_scores.items()})
    total = sum(raw.values())
    if total <= 0:
        return {key: 1.0 / len(raw) for key in raw}
    return {key: value / total for key, value in raw.items()}


def top_with_margin(scores):
    ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    top_value = float(ranked[0][1])
    margin = top_value - (float(ranked[1][1]) if len(ranked) > 1 else 0.0)
    return int(ranked[0][0]), top_value, margin


def score_sample(active_model, row, image_root=TRAIN_IMAGE_DIR, has_gold=True):
    gold_order = order_to_sequence(row["Answer_list"]) if has_gold else None
    sample_id = str(row["Id"])

    endpoint_examples = [
        make_eval_example(row, "first", image_root=image_root),
        make_eval_example(row, "last", image_root=image_root),
    ]
    first_probs, last_probs = score_next_token_examples(active_model, endpoint_examples, DIGIT_TOKEN_IDS, batch_size=2)
    first_probs = {str(k): v for k, v in first_probs.items()}
    last_probs = {str(k): v for k, v in last_probs.items()}

    pair_requests = []
    pair_examples = []
    for i, j in PAIR_INDICES:
        a, b = i + 1, j + 1
        for left, right in [(a, b), (b, a)]:
            pair_requests.append((left, right))
            pair_examples.append(make_eval_example(row, "pairwise", pair=(left, right), image_root=image_root))
    pair_outputs = score_next_token_examples(active_model, pair_examples, AB_TOKEN_IDS, batch_size=NEXT_TOKEN_BATCH_SIZE)
    pair_lookup = {pair: output for pair, output in zip(pair_requests, pair_outputs)}

    pair_probs = {}
    for i, j in PAIR_INDICES:
        a, b = i + 1, j + 1
        p_forward = pair_lookup[(a, b)]["A"]
        p_reverse = pair_lookup[(b, a)]["A"]
        combined = combine_bidirectional_pair(p_forward, p_reverse)
        pair_probs[f"{a}>{b}"] = {"forward_prob": float(p_forward), "reverse_prob": float(p_reverse), "combined_prob": float(combined), "swap_inconsistency": float(abs(p_forward + p_reverse - 1.0))}
        pair_probs[f"{b}>{a}"] = {"forward_prob": float(1.0 - p_forward), "reverse_prob": float(1.0 - p_reverse), "combined_prob": float(1.0 - combined), "swap_inconsistency": float(abs(p_forward + p_reverse - 1.0))}

    order_scores = score_sequence_candidates(active_model, make_eval_example(row, "order", image_root=image_root), PERMUTATIONS)
    order_probs = softmax_scores(order_scores)
    best_order_key = max(order_scores, key=order_scores.get)
    teacher_forced_order = list(parse_order_key(best_order_key))

    if RUN_DIRECT_GENERATION:
        direct_text = generate_text(active_model, make_eval_example(row, "order", image_root=image_root))
        direct_order = parse_compact_order(direct_text, 4)
    else:
        direct_text = best_order_key
        direct_order = teacher_forced_order

    return {
        "sample_id": sample_id,
        "gold_order": gold_order,
        "direct_order": direct_order,
        "direct_text": direct_text,
        "first_probs": first_probs,
        "last_probs": last_probs,
        "bidirectional_pair_probs": pair_probs,
        "order_24_scores": order_scores,
        "order_24_probs": order_probs,
    }


def conditional_decode(active_model, row, scored, image_root=TRAIN_IMAGE_DIR):
    frames = [1, 2, 3, 4]
    order_probs = scored["order_24_probs"]
    pair_probs = scored["bidirectional_pair_probs"]
    order_first = position_marginal(order_probs, 0, frames)
    order_last = position_marginal(order_probs, 3, frames)
    first_head = normalized({int(k): v for k, v in scored["first_probs"].items()})
    last_head = normalized({int(k): v for k, v in scored["last_probs"].items()})
    pair_first = endpoint_probs_from_pair(pair_probs, frames, "first")
    pair_last = endpoint_probs_from_pair(pair_probs, frames, "last")
    raw_first = fuse_three_raw(order_first, first_head, pair_first, 0.45, 0.20, 0.35)
    raw_last = fuse_three_raw(order_last, last_head, pair_last, 0.45, 0.20, 0.35)
    fused_first = normalized(raw_first)
    fused_last = normalized(raw_last)
    endpoint_slot_probs = normalize_endpoint_slots(raw_first, raw_last)
    first_candidate, first_top1, first_margin = top_with_margin(fused_first)
    last_candidate, last_top1, last_margin = top_with_margin(fused_last)
    first_priority = endpoint_slot_probs[f"first:{first_candidate}"]
    last_priority = endpoint_slot_probs[f"last:{last_candidate}"]

    if first_priority >= last_priority:
        first = first_candidate
        remaining = [x for x in frames if x != first]
        candidates = list(itertools.permutations(remaining))
        cond_scores = score_sequence_candidates(active_model, make_eval_example(row, "fixed_first", fixed_first=first, image_root=image_root), candidates)
        cond_probs = softmax_scores(cond_scores)
        conditional_order_last = position_marginal(cond_probs, -1, remaining)
        conditional_fused = fuse_three(conditional_order_last, normalized({i: float(scored["last_probs"][str(i)]) for i in remaining}), endpoint_probs_from_pair(pair_probs, remaining, "last"), 0.45, 0.15, 0.40)
        last = max(conditional_fused, key=conditional_fused.get)
        fixed_first_selected = True
    else:
        last = last_candidate
        remaining = [x for x in frames if x != last]
        candidates = list(itertools.permutations(remaining))
        cond_scores = score_sequence_candidates(active_model, make_eval_example(row, "fixed_last", fixed_last=last, image_root=image_root), candidates)
        cond_probs = softmax_scores(cond_scores)
        conditional_order_first = position_marginal(cond_probs, 0, remaining)
        conditional_fused = fuse_three(conditional_order_first, normalized({i: float(scored["first_probs"][str(i)]) for i in remaining}), endpoint_probs_from_pair(pair_probs, remaining, "first"), 0.45, 0.15, 0.40)
        first = max(conditional_fused, key=conditional_fused.get)
        fixed_first_selected = False

    middle = [x for x in frames if x not in {first, last}]
    middle_candidates = list(itertools.permutations(middle))
    middle_scores = score_sequence_candidates(active_model, make_eval_example(row, "fixed_endpoints", fixed_first=first, fixed_last=last, image_root=image_root), middle_candidates)
    middle_probs = softmax_scores(middle_scores)
    a, b = middle
    order_a_before_b = middle_probs.get(f"{a} {b}", 0.0)
    pair_a_before_b = pair_probs[f"{a}>{b}"]["combined_prob"]
    score_a_before_b = 0.60 * order_a_before_b + 0.40 * pair_a_before_b
    middle_order = [a, b] if score_a_before_b >= 0.5 else [b, a]
    final_order = [first, *middle_order, last]
    scored.update({
        "raw_fused_first": raw_first,
        "raw_fused_last": raw_last,
        "fused_first": fused_first,
        "fused_last": fused_last,
        "endpoint_slot_probs": endpoint_slot_probs,
        "first_top1": first_top1,
        "last_top1": last_top1,
        "first_margin": first_margin,
        "last_margin": last_margin,
        "first_priority": first_priority,
        "last_priority": last_priority,
        "first_endpoint_selected": fixed_first_selected,
        "fixed_endpoint": first if fixed_first_selected else last,
        "selected_other_endpoint": last if fixed_first_selected else first,
        "conditional_order_6_probs": cond_probs,
        "conditional_middle_probs": middle_probs,
        "final_order": final_order,
    })
    return scored


def order_metric_row(pred_order, gold_order):
    if pred_order is None or gold_order is None:
        return {"exact": 0.0, "first": 0.0, "last": 0.0, "both_endpoints": 0.0, "position": 0.0, "relative_pair": 0.0}
    ranks_p = {x: i for i, x in enumerate(pred_order)}
    ranks_g = {x: i for i, x in enumerate(gold_order)}
    return {
        "exact": float(pred_order == gold_order),
        "first": float(pred_order[0] == gold_order[0]),
        "last": float(pred_order[-1] == gold_order[-1]),
        "both_endpoints": float(pred_order[0] == gold_order[0] and pred_order[-1] == gold_order[-1]),
        "position": float(np.mean([p == g for p, g in zip(pred_order, gold_order)])),
        "relative_pair": float(np.mean([(ranks_p[a] < ranks_p[b]) == (ranks_g[a] < ranks_g[b]) for a, b in itertools.combinations([1, 2, 3, 4], 2)])),
    }


In [ ]:
# 4.5) KV-cache token-level diagnostic before full eval
# Run this after cells 1-4. It loads checkpoint-1200 and compares one candidate token by token.
RUN_KV_TOKEN_DIAGNOSTIC = True
KV_DIAGNOSTIC_SAMPLE_INDEX = 0
KV_DIAGNOSTIC_ORDER = (1, 2, 3, 4)


def inspect_generation_helpers(active_model):
    import inspect
    import transformers
    base_model = active_model.get_base_model() if hasattr(active_model, "get_base_model") else active_model
    print("transformers:", transformers.__version__)
    print("model class:", type(active_model))
    print("base class:", type(base_model))
    print("prepare_inputs_for_generation signature:")
    print(inspect.signature(base_model.prepare_inputs_for_generation))
    try:
        print(inspect.getsource(base_model.prepare_inputs_for_generation))
    except Exception as exc:
        print("Could not print prepare_inputs_for_generation source:", repr(exc))


@torch.no_grad()
def recompute_token_logprobs(active_model, example, order):
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "right"
    try:
        prompt = processor.apply_chat_template(
            make_messages(example, include_answer=False),
            tokenize=False,
            add_generation_prompt=True,
        )
        answer_text = compact_order(order)
        images = [load_rgb(path) for path in example["image_paths"]]
        prompt_inputs = processor(text=[prompt], images=[images], return_tensors="pt")
        prompt_len = int(prompt_inputs["attention_mask"][0].sum().item())
        inputs = processor(text=[prompt + answer_text], images=[images], return_tensors="pt")
        inputs = _batch_to_device(inputs, active_model)
        outputs = active_model(**inputs)
        input_ids = inputs["input_ids"][0]
        attention_len = int(inputs["attention_mask"][0].sum().item())
        target_len = attention_len - prompt_len
        target_ids = input_ids[prompt_len:prompt_len + target_len]
        logits = outputs.logits[0, prompt_len - 1:prompt_len - 1 + target_len]
        log_probs = torch.log_softmax(logits.float(), dim=-1)
        token_log_probs = log_probs.gather(1, target_ids[:, None]).squeeze(1)
        return {
            "prompt": prompt,
            "answer_text": answer_text,
            "prompt_len": prompt_len,
            "target_ids": target_ids.detach().cpu().tolist(),
            "target_tokens": processor.tokenizer.convert_ids_to_tokens(target_ids.detach().cpu().tolist()),
            "token_log_probs": token_log_probs.detach().cpu().tolist(),
            "logits": logits.detach().float().cpu(),
        }
    finally:
        processor.tokenizer.padding_side = old_padding_side


@torch.no_grad()
def kv_token_logprobs_current(active_model, example, order):
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "right"
    prompt_cache = None
    base_cache_length = None
    try:
        device = model_device(active_model)
        prompt = processor.apply_chat_template(
            make_messages(example, include_answer=False),
            tokenize=False,
            add_generation_prompt=True,
        )
        answer_text = compact_order(order)
        images = [load_rgb(path) for path in example["image_paths"]]
        prompt_inputs = processor(text=[prompt], images=[images], return_tensors="pt")
        prompt_inputs = _batch_to_device(prompt_inputs, active_model)
        prompt_outputs = active_model(**prompt_inputs, use_cache=True)
        prompt_cache = prompt_outputs.past_key_values
        if prompt_cache is None:
            raise RuntimeError("No past_key_values returned")
        if not hasattr(prompt_cache, "crop") or not hasattr(prompt_cache, "get_seq_length"):
            raise RuntimeError(f"Unsupported cache type: {type(prompt_cache)}")
        base_cache_length = int(prompt_cache.get_seq_length())
        prompt_attention_mask = prompt_inputs["attention_mask"]
        prompt_len = int(prompt_attention_mask[0].sum().item())
        first_logits = prompt_outputs.logits[0, prompt_len - 1]
        prompt_rope_deltas = getattr(prompt_outputs, "rope_deltas", None)
        target_ids = _target_ids_after_prompt(prompt, answer_text, device)
        candidate_logits = [first_logits]
        if target_ids.numel() > 1:
            prompt_cache.crop(base_cache_length)
            previous_ids = target_ids[:-1].unsqueeze(0)
            continuation_mask = torch.ones(
                (1, previous_ids.shape[1]),
                dtype=prompt_attention_mask.dtype,
                device=device,
            )
            attention_mask = torch.cat([prompt_attention_mask, continuation_mask], dim=1)
            cache_position = torch.arange(
                base_cache_length,
                base_cache_length + previous_ids.shape[1],
                dtype=torch.long,
                device=device,
            )
            position_ids = build_qwen2vl_continuation_position_ids(
                prompt_attention_mask=prompt_attention_mask,
                rope_deltas=prompt_rope_deltas,
                continuation_length=previous_ids.shape[1],
                device=device,
            )
            print("prompt rope_deltas:", prompt_rope_deltas)
            print("manual position_ids:", position_ids[:, 0, :])
            print("cache_position:", cache_position)
            print("previous tokens:", processor.tokenizer.convert_ids_to_tokens(previous_ids[0].detach().cpu().tolist()))
            kwargs = {
                "input_ids": previous_ids,
                "attention_mask": attention_mask,
                "position_ids": position_ids,
                "past_key_values": prompt_cache,
                "cache_position": cache_position,
                "use_cache": True,
                "return_dict": True,
                "pixel_values": None,
                "pixel_values_videos": None,
                "image_grid_thw": None,
                "video_grid_thw": None,
            }
            outputs = active_model(**kwargs)
            candidate_logits.extend(outputs.logits[0, idx] for idx in range(outputs.logits.shape[1]))
        logits = torch.stack(candidate_logits[:target_ids.numel()], dim=0)
        log_probs = torch.log_softmax(logits.float(), dim=-1)
        token_log_probs = log_probs.gather(1, target_ids[:, None]).squeeze(1)
        return {
            "prompt_len": prompt_len,
            "cache_length": base_cache_length,
            "target_ids": target_ids.detach().cpu().tolist(),
            "target_tokens": processor.tokenizer.convert_ids_to_tokens(target_ids.detach().cpu().tolist()),
            "token_log_probs": token_log_probs.detach().cpu().tolist(),
            "logits": logits.detach().float().cpu(),
        }
    finally:
        if prompt_cache is not None and base_cache_length is not None and hasattr(prompt_cache, "crop"):
            prompt_cache.crop(base_cache_length)
        processor.tokenizer.padding_side = old_padding_side


def run_kv_token_diagnostic(sample_index=KV_DIAGNOSTIC_SAMPLE_INDEX, order=KV_DIAGNOSTIC_ORDER):
    checkpoint_dirs = find_checkpoint_dirs()
    matches = [path for path in checkpoint_dirs if checkpoint_name(path) == "checkpoint-1200"]
    if not matches:
        raise FileNotFoundError("checkpoint-1200 not found")
    active_model = load_eval_model(matches[0])
    inspect_generation_helpers(active_model)

    row = validation_df.sample(n=min(FULL_EVAL_ROWS, len(validation_df)), random_state=SEED + 1).reset_index(drop=True).iloc[int(sample_index)]
    example = make_eval_example(row, "order", image_root=TRAIN_IMAGE_DIR)
    print("sample_id:", row["Id"])
    print("order:", order)

    recompute = recompute_token_logprobs(active_model, example, order)
    kv = kv_token_logprobs_current(active_model, example, order)
    if recompute["target_ids"] != kv["target_ids"]:
        print("target id mismatch")
        print("recompute:", recompute["target_ids"], recompute["target_tokens"])
        print("kv       :", kv["target_ids"], kv["target_tokens"])

    rows = []
    for idx, token_id in enumerate(recompute["target_ids"]):
        recompute_lp = float(recompute["token_log_probs"][idx])
        kv_lp = float(kv["token_log_probs"][idx]) if idx < len(kv["token_log_probs"]) else float("nan")
        logit_diff = float(torch.max(torch.abs(recompute["logits"][idx] - kv["logits"][idx])).item()) if idx < kv["logits"].shape[0] else float("nan")
        rows.append({
            "token_index": idx,
            "token_id": int(token_id),
            "token": recompute["target_tokens"][idx],
            "recompute_logprob": recompute_lp,
            "kv_logprob": kv_lp,
            "abs_logprob_diff": abs(recompute_lp - kv_lp),
            "max_abs_logit_diff": logit_diff,
        })
    df = pd.DataFrame(rows)
    display(df)
    out_path = EVAL_DIR / "checkpoint1200_kv_token_diagnostic.csv"
    df.to_csv(out_path, index=False)
    print("saved:", out_path)
    print("prompt_len recompute/kv:", recompute["prompt_len"], kv["prompt_len"])
    print("cache_length:", kv["cache_length"])
    return df


if RUN_KV_TOKEN_DIAGNOSTIC:
    kv_token_diagnostic_df = run_kv_token_diagnostic()


In [ ]:
# 5) Evaluate checkpoint-1200 with dual-branch pair-priority decoder
DECODER_TAG = (
    "init10_30_60"
    "_dualbranch"
    "_second10_30_60"
    "_middle30_70"
    "_anchor50_50"
    "_rerank10_25_55_10"
    "_manual_mrope_kv_decode_v6"
)

INITIAL_WEIGHTS = (0.10, 0.30, 0.60)  # order, endpoint head, pair
SECOND_WEIGHTS = (0.10, 0.30, 0.60)   # conditional, endpoint head, pair
MIDDLE_WEIGHTS = (0.30, 0.70)         # conditional middle, pair
TOP1_WEIGHT = 0.50
MARGIN_WEIGHT = 0.50
FINAL_BRANCH_WEIGHTS = {
    "order": 0.10,
    "endpoint": 0.25,
    "pair": 0.55,
    "anchor": 0.10,
}
FRAMES = [1, 2, 3, 4]
KV_CACHE_VALIDATION_ROWS = 3
KV_CACHE_VALIDATED = False
KV_CACHE_VALIDATION_MAX_ABS_DIFF = 0.05


def normalize_two(a, b, eps=1e-8):
    a = max(float(a), 0.0)
    b = max(float(b), 0.0)
    total = a + b + eps
    return a / total, b / total


def pair_prob(pair_probs, a, b):
    return float(pair_probs[f"{int(a)}>{int(b)}"]["combined_prob"])


def pair_order_score(order, pair_probs):
    return float(np.mean([
        pair_prob(pair_probs, order[i], order[j])
        for i in range(4)
        for j in range(i + 1, 4)
    ]))


def validate_kv_cache_sequence_scoring(active_model, rows, n=KV_CACHE_VALIDATION_ROWS):
    global USE_KV_CACHE_SEQUENCE_SCORING, KV_CACHE_VALIDATED
    if KV_CACHE_VALIDATED:
        return []

    validation_rows = rows.head(min(int(n), len(rows))).reset_index(drop=True)
    reports = []
    old_flag = USE_KV_CACHE_SEQUENCE_SCORING
    print(f"Validating KV-cache sequence scoring on {len(validation_rows)} samples...")

    try:
        for row_index, row in validation_rows.iterrows():
            examples = [("order24", make_eval_example(row, "order", image_root=TRAIN_IMAGE_DIR), PERMUTATIONS)]
            gold_order = order_to_sequence(row["Answer_list"])
            first = gold_order[0]
            last = gold_order[-1]
            remaining_first = [x for x in FRAMES if x != first]
            remaining_middle = [x for x in FRAMES if x not in {first, last}]
            examples.append(("fixed_first6", make_eval_example(row, "fixed_first", fixed_first=first, image_root=TRAIN_IMAGE_DIR), list(itertools.permutations(remaining_first))))
            examples.append(("fixed_endpoints2", make_eval_example(row, "fixed_endpoints", fixed_first=first, fixed_last=last, image_root=TRAIN_IMAGE_DIR), list(itertools.permutations(remaining_middle))))

            for name, example, candidates in examples:
                USE_KV_CACHE_SEQUENCE_SCORING = True
                kv_scores = score_sequence_candidates(active_model, example, candidates)
                USE_KV_CACHE_SEQUENCE_SCORING = False
                recompute_scores = score_sequence_candidates(active_model, example, candidates)
                keys = sorted(set(kv_scores) & set(recompute_scores))
                same_top1 = max(kv_scores, key=kv_scores.get) == max(recompute_scores, key=recompute_scores.get)
                max_abs_diff = max(abs(float(kv_scores[k]) - float(recompute_scores[k])) for k in keys) if keys else float("inf")
                report = {
                    "sample_id": str(row["Id"]),
                    "check": name,
                    "same_top1": bool(same_top1),
                    "max_abs_diff": float(max_abs_diff),
                    "kv_top1": max(kv_scores, key=kv_scores.get),
                    "recompute_top1": max(recompute_scores, key=recompute_scores.get),
                }
                reports.append(report)
                print(report)
    finally:
        USE_KV_CACHE_SEQUENCE_SCORING = old_flag

    report_df = pd.DataFrame(reports)
    report_path = EVAL_DIR / f"checkpoint1200_dualbranch_{DECODER_TAG}_kv_validation.csv"
    report_df.to_csv(report_path, index=False)
    print("saved KV validation:", report_path)

    failed = report_df[(~report_df["same_top1"]) | (report_df["max_abs_diff"] > KV_CACHE_VALIDATION_MAX_ABS_DIFF)] if len(report_df) else pd.DataFrame()
    if len(failed):
        display(failed)
        raise RuntimeError(
            "Safe KV scoring still differs from recompute. "
            "Stop here instead of falling back to a multi-hour recompute run."
        )
    KV_CACHE_VALIDATED = True
    return reports


def choose_middle_pair_priority(active_model, row, pair_probs, first, last, image_root=TRAIN_IMAGE_DIR):
    middle = [x for x in FRAMES if x not in {int(first), int(last)}]
    if len(middle) != 2:
        return middle, {}, {}

    candidates = list(itertools.permutations(middle))
    middle_scores = score_sequence_candidates(
        active_model,
        make_eval_example(row, "fixed_endpoints", fixed_first=first, fixed_last=last, image_root=image_root),
        candidates,
    )
    middle_probs = softmax_scores(middle_scores)

    a, b = middle
    ab_key = f"{a} {b}"
    ba_key = f"{b} {a}"
    cond_w, pair_w = MIDDLE_WEIGHTS
    ab_score = cond_w * float(middle_probs.get(ab_key, 0.0)) + pair_w * pair_prob(pair_probs, a, b)
    ba_score = cond_w * float(middle_probs.get(ba_key, 0.0)) + pair_w * pair_prob(pair_probs, b, a)
    middle_order = [a, b] if ab_score >= ba_score else [b, a]
    middle_choice_scores = {ab_key: float(ab_score), ba_key: float(ba_score)}
    return middle_order, middle_probs, middle_choice_scores


def choose_middle_cached(active_model, row, pair_probs, first, last, middle_cache, image_root=TRAIN_IMAGE_DIR):
    key = (int(first), int(last))
    if key not in middle_cache:
        middle_cache[key] = choose_middle_pair_priority(active_model, row, pair_probs, first, last, image_root=image_root)
    return middle_cache[key]


def decode_first_fixed_branch(active_model, row, scored, first_candidate, middle_cache, image_root=TRAIN_IMAGE_DIR):
    pair_probs = scored["bidirectional_pair_probs"]
    first = int(first_candidate)
    remaining = [x for x in FRAMES if x != first]
    cond_candidates = list(itertools.permutations(remaining))
    cond_scores = score_sequence_candidates(
        active_model,
        make_eval_example(row, "fixed_first", fixed_first=first, image_root=image_root),
        cond_candidates,
    )
    cond_probs = softmax_scores(cond_scores)
    conditional_last = position_marginal(cond_probs, -1, remaining)
    endpoint_last = normalized({i: float(scored["last_probs"][str(i)]) for i in remaining})
    pair_last = endpoint_probs_from_pair(pair_probs, remaining, "last")
    sw_cond, sw_endpoint, sw_pair = SECOND_WEIGHTS
    second_last_scores = normalized({
        j: sw_cond * conditional_last[j] + sw_endpoint * endpoint_last[j] + sw_pair * pair_last[j]
        for j in remaining
    })
    last = max(second_last_scores, key=second_last_scores.get)
    middle_order, middle_probs, middle_choice_scores = choose_middle_cached(
        active_model, row, pair_probs, first, last, middle_cache, image_root=image_root
    )
    final_order = [first, *middle_order, int(last)]
    return {
        "name": "first_fixed",
        "fixed_endpoint": first,
        "second_endpoint": int(last),
        "second_endpoint_scores": second_last_scores,
        "conditional_order_6_probs": cond_probs,
        "conditional_middle_probs": middle_probs,
        "middle_choice_scores": middle_choice_scores,
        "final_order": final_order,
    }


def decode_last_fixed_branch(active_model, row, scored, last_candidate, middle_cache, image_root=TRAIN_IMAGE_DIR):
    pair_probs = scored["bidirectional_pair_probs"]
    last = int(last_candidate)
    remaining = [x for x in FRAMES if x != last]
    cond_candidates = list(itertools.permutations(remaining))
    cond_scores = score_sequence_candidates(
        active_model,
        make_eval_example(row, "fixed_last", fixed_last=last, image_root=image_root),
        cond_candidates,
    )
    cond_probs = softmax_scores(cond_scores)
    conditional_first = position_marginal(cond_probs, 0, remaining)
    endpoint_first = normalized({i: float(scored["first_probs"][str(i)]) for i in remaining})
    pair_first = endpoint_probs_from_pair(pair_probs, remaining, "first")
    sw_cond, sw_endpoint, sw_pair = SECOND_WEIGHTS
    second_first_scores = normalized({
        j: sw_cond * conditional_first[j] + sw_endpoint * endpoint_first[j] + sw_pair * pair_first[j]
        for j in remaining
    })
    first = max(second_first_scores, key=second_first_scores.get)
    middle_order, middle_probs, middle_choice_scores = choose_middle_cached(
        active_model, row, pair_probs, first, last, middle_cache, image_root=image_root
    )
    final_order = [int(first), *middle_order, last]
    return {
        "name": "last_fixed",
        "fixed_endpoint": last,
        "second_endpoint": int(first),
        "second_endpoint_scores": second_first_scores,
        "conditional_order_6_probs": cond_probs,
        "conditional_middle_probs": middle_probs,
        "middle_choice_scores": middle_choice_scores,
        "final_order": final_order,
    }


def branch_component_scores(branch_name, order, scored, fused_first, fused_last, first_anchor, last_anchor):
    order_key = " ".join(map(str, order))
    order_score = float(scored["order_24_probs"].get(order_key, 0.0))
    endpoint_score = (float(fused_first.get(order[0], 0.0)) + float(fused_last.get(order[-1], 0.0))) / 2.0
    pair_score = pair_order_score(order, scored["bidirectional_pair_probs"])
    anchor_score = float(first_anchor if branch_name == "first_fixed" else last_anchor)
    return {
        "order": order_score,
        "endpoint": endpoint_score,
        "pair": pair_score,
        "anchor": anchor_score,
    }


def rerank_two_branches(first_components, last_components):
    first_norm = {}
    last_norm = {}
    for key in ["order", "endpoint", "pair", "anchor"]:
        first_norm[key], last_norm[key] = normalize_two(first_components[key], last_components[key])
    first_score = sum(FINAL_BRANCH_WEIGHTS[key] * first_norm[key] for key in FINAL_BRANCH_WEIGHTS)
    last_score = sum(FINAL_BRANCH_WEIGHTS[key] * last_norm[key] for key in FINAL_BRANCH_WEIGHTS)
    return float(first_score), float(last_score), first_norm, last_norm


def dual_branch_decode(active_model, row, scored, image_root=TRAIN_IMAGE_DIR):
    order_probs = scored["order_24_probs"]
    pair_probs = scored["bidirectional_pair_probs"]

    order_first = position_marginal(order_probs, 0, FRAMES)
    order_last = position_marginal(order_probs, 3, FRAMES)
    first_head = normalized({int(k): v for k, v in scored["first_probs"].items()})
    last_head = normalized({int(k): v for k, v in scored["last_probs"].items()})
    pair_first = endpoint_probs_from_pair(pair_probs, FRAMES, "first")
    pair_last = endpoint_probs_from_pair(pair_probs, FRAMES, "last")

    iw_order, iw_endpoint, iw_pair = INITIAL_WEIGHTS
    raw_first = fuse_three_raw(order_first, first_head, pair_first, iw_order, iw_endpoint, iw_pair)
    raw_last = fuse_three_raw(order_last, last_head, pair_last, iw_order, iw_endpoint, iw_pair)
    fused_first = normalized(raw_first)
    fused_last = normalized(raw_last)

    first_candidate, first_top1, first_margin = top_with_margin(fused_first)
    last_candidate, last_top1, last_margin = top_with_margin(fused_last)
    first_anchor_confidence = TOP1_WEIGHT * first_top1 + MARGIN_WEIGHT * first_margin
    last_anchor_confidence = TOP1_WEIGHT * last_top1 + MARGIN_WEIGHT * last_margin

    middle_cache = {}
    first_branch = decode_first_fixed_branch(active_model, row, scored, first_candidate, middle_cache, image_root=image_root)
    last_branch = decode_last_fixed_branch(active_model, row, scored, last_candidate, middle_cache, image_root=image_root)

    first_components = branch_component_scores(
        "first_fixed", first_branch["final_order"], scored, fused_first, fused_last,
        first_anchor_confidence, last_anchor_confidence,
    )
    last_components = branch_component_scores(
        "last_fixed", last_branch["final_order"], scored, fused_first, fused_last,
        first_anchor_confidence, last_anchor_confidence,
    )
    first_branch_score, last_branch_score, first_norm, last_norm = rerank_two_branches(first_components, last_components)

    if first_branch_score >= last_branch_score:
        selected_branch = "first_fixed"
        final_order = first_branch["final_order"]
    else:
        selected_branch = "last_fixed"
        final_order = last_branch["final_order"]

    decoded = dict(scored)
    decoded.update({
        "decoder_tag": DECODER_TAG,
        "initial_weights": INITIAL_WEIGHTS,
        "second_weights": SECOND_WEIGHTS,
        "middle_weights": MIDDLE_WEIGHTS,
        "final_branch_weights": FINAL_BRANCH_WEIGHTS,
        "fused_first": fused_first,
        "fused_last": fused_last,
        "first_top1": first_top1,
        "first_margin": first_margin,
        "first_anchor_confidence": first_anchor_confidence,
        "last_top1": last_top1,
        "last_margin": last_margin,
        "last_anchor_confidence": last_anchor_confidence,
        "middle_cache_keys": [list(key) for key in middle_cache],
        "first_branch": {
            **first_branch,
            "component_scores": first_components,
            "normalized_component_scores": first_norm,
            "branch_score": first_branch_score,
        },
        "last_branch": {
            **last_branch,
            "component_scores": last_components,
            "normalized_component_scores": last_norm,
            "branch_score": last_branch_score,
        },
        "selected_branch": selected_branch,
        "final_order": final_order,
    })
    return decoded


def evaluate_dualbranch_checkpoint(adapter_dir, rows, tag, image_root=TRAIN_IMAGE_DIR, has_gold=True):
    ckpt = checkpoint_name(adapter_dir)
    pred_dir = EVAL_DIR / "sample_predictions"
    pred_dir.mkdir(parents=True, exist_ok=True)
    cache_path = pred_dir / f"{ckpt}_{tag}_{DECODER_TAG}.json"
    if cache_path.exists():
        print("loading cached:", cache_path)
        with open(cache_path, "r", encoding="utf-8") as f:
            records = json.load(f)
    else:
        active_model = load_eval_model(adapter_dir)
        validate_kv_cache_sequence_scoring(active_model, rows, n=KV_CACHE_VALIDATION_ROWS)
        records = []
        for _, row in tqdm(rows.iterrows(), total=len(rows), desc=f"{ckpt} {tag} {DECODER_TAG}"):
            scored = score_sample(active_model, row, image_root=image_root, has_gold=has_gold)
            decoded = dual_branch_decode(active_model, row, scored, image_root=image_root)
            records.append(decoded)
        with open(cache_path, "w", encoding="utf-8") as f:
            json.dump(records, f, ensure_ascii=False, indent=2)
        print("saved records:", cache_path)

    if not has_gold:
        return {}, records, cache_path, None

    metric_rows = []
    for record in records:
        gold = record["gold_order"]
        final = order_metric_row(record["final_order"], gold)
        direct = order_metric_row(record["direct_order"], gold)
        first_branch_metric = order_metric_row(record["first_branch"]["final_order"], gold)
        last_branch_metric = order_metric_row(record["last_branch"]["final_order"], gold)
        first_anchor_correct = float(record["first_branch"]["fixed_endpoint"] == gold[0])
        first_second_correct = float(record["first_branch"]["second_endpoint"] == gold[-1])
        last_anchor_correct = float(record["last_branch"]["fixed_endpoint"] == gold[-1])
        last_second_correct = float(record["last_branch"]["second_endpoint"] == gold[0])
        first_both_endpoints = float(first_anchor_correct and first_second_correct)
        last_both_endpoints = float(last_anchor_correct and last_second_correct)
        metric_rows.append({
            "sample_id": record["sample_id"],
            "gold_order": " ".join(map(str, gold)),
            "final_order": " ".join(map(str, record["final_order"])),
            "first_branch_order": " ".join(map(str, record["first_branch"]["final_order"])),
            "last_branch_order": " ".join(map(str, record["last_branch"]["final_order"])),
            "selected_branch": record["selected_branch"],
            "direct_exact": direct["exact"],
            "reranker_exact": final["exact"],
            "reranker_first": final["first"],
            "reranker_last": final["last"],
            "reranker_both_endpoints": final["both_endpoints"],
            "reranker_position": final["position"],
            "reranker_relative_pair": final["relative_pair"],
            "first_branch_exact": first_branch_metric["exact"],
            "last_branch_exact": last_branch_metric["exact"],
            "branch_oracle_exact": float(first_branch_metric["exact"] or last_branch_metric["exact"]),
            "branch_disagree": float(record["first_branch"]["final_order"] != record["last_branch"]["final_order"]),
            "first_only_exact": float(first_branch_metric["exact"] and not last_branch_metric["exact"]),
            "last_only_exact": float(last_branch_metric["exact"] and not first_branch_metric["exact"]),
            "both_branch_exact": float(first_branch_metric["exact"] and last_branch_metric["exact"]),
            "selected_first_branch": float(record["selected_branch"] == "first_fixed"),
            "selected_branch_correct": float(
                (record["selected_branch"] == "first_fixed" and first_branch_metric["exact"]) or
                (record["selected_branch"] == "last_fixed" and last_branch_metric["exact"])
            ),
            "first_anchor_correct": first_anchor_correct,
            "first_second_correct": first_second_correct,
            "first_second_given_anchor": first_second_correct if first_anchor_correct else np.nan,
            "first_both_endpoints": first_both_endpoints,
            "first_middle_exact_given_endpoints": first_branch_metric["exact"] if first_both_endpoints else np.nan,
            "last_anchor_correct": last_anchor_correct,
            "last_second_correct": last_second_correct,
            "last_second_given_anchor": last_second_correct if last_anchor_correct else np.nan,
            "last_both_endpoints": last_both_endpoints,
            "last_middle_exact_given_endpoints": last_branch_metric["exact"] if last_both_endpoints else np.nan,
            "first_branch_score": record["first_branch"]["branch_score"],
            "last_branch_score": record["last_branch"]["branch_score"],
            "first_anchor_confidence": record["first_anchor_confidence"],
            "last_anchor_confidence": record["last_anchor_confidence"],
            "middle_cache_entries": len(record.get("middle_cache_keys", [])),
            "bidirectional_pair_accuracy": float(np.mean([
                ((record["bidirectional_pair_probs"][f"{a}>{b}"]["combined_prob"] >= 0.5) == ({x: i for i, x in enumerate(gold)}[a] < {x: i for i, x in enumerate(gold)}[b]))
                for a, b in itertools.combinations(FRAMES, 2)
            ])),
        })
    detail_df = pd.DataFrame(metric_rows)
    summary = {col: float(detail_df[col].mean()) for col in detail_df.columns if col not in {"sample_id", "gold_order", "final_order", "first_branch_order", "last_branch_order", "selected_branch"}}
    detail_path = EVAL_DIR / f"checkpoint1200_dualbranch_{DECODER_TAG}_detail.csv"
    detail_df.to_csv(detail_path, index=False)
    summary.update({
        "checkpoint": ckpt,
        "tag": tag,
        "decoder_tag": DECODER_TAG,
        "adapter_dir": str(adapter_dir),
        "rows": len(detail_df),
        "cache_path": str(cache_path),
        "detail_path": str(detail_path),
    })
    return summary, records, cache_path, detail_df


checkpoint_dirs = find_checkpoint_dirs()
selected = set(SELECTED_CHECKPOINTS)
checkpoint_dirs = [path for path in checkpoint_dirs if checkpoint_name(path) in selected]
missing = sorted(selected - {checkpoint_name(path) for path in checkpoint_dirs})
if missing:
    print("missing selected checkpoints:", missing)
print("selected checkpoints:", [checkpoint_name(path) for path in checkpoint_dirs])
if not checkpoint_dirs:
    raise FileNotFoundError("No selected checkpoints found.")

full_rows = validation_df.sample(
    n=min(FULL_EVAL_ROWS, len(validation_df)),
    random_state=SEED + 1,
).reset_index(drop=True)

dual_summaries = []
dual_details = []
for adapter_dir in checkpoint_dirs:
    summary, records, cache_path, detail_df = evaluate_dualbranch_checkpoint(
        adapter_dir,
        full_rows,
        f"full{len(full_rows)}",
        image_root=TRAIN_IMAGE_DIR,
        has_gold=True,
    )
    dual_summaries.append(summary)
    if detail_df is not None:
        dual_details.append(detail_df.assign(checkpoint=checkpoint_name(adapter_dir)))

summary_df = pd.DataFrame(dual_summaries).sort_values(
    [
        "reranker_exact",
        "reranker_both_endpoints",
        "reranker_relative_pair",
        "reranker_position",
        "branch_oracle_exact",
        "direct_exact",
    ],
    ascending=False,
).reset_index(drop=True)

summary_path = EVAL_DIR / f"checkpoint1200_dualbranch_{DECODER_TAG}_summary.csv"
summary_df.to_csv(summary_path, index=False)
display(summary_df)
print("saved summary:", summary_path)
if dual_details:
    all_detail_path = EVAL_DIR / f"checkpoint1200_dualbranch_{DECODER_TAG}_all_detail.csv"
    pd.concat(dual_details, ignore_index=True).to_csv(all_detail_path, index=False)
    print("saved all detail:", all_detail_path)
